# Day I21 — 캡스톤: FOG 탐지 미니 프로젝트 보고서와 5분 발표

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/I-gait/day-I21.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/I-gait/day-I21.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 보고서용 참가자별 표와 참가자 부트스트랩 구간

Day I19 예시 2가 저장한 `results/advanced/I/I19_loso_pred.csv.gz`를 읽습니다. 추가 패키지 없이 Markdown 표를 직접 만듭니다.

In [ ]:
import numpy as np
import pandas as pd

win = pd.read_csv("results/advanced/I/I19_loso_pred.csv.gz")      # Day I19 예시 2의 출력
rng = np.random.default_rng(21)
rows = []
for s, g in win.groupby("subject"):
    y = g["fog"].to_numpy().astype(bool)
    pred = g["p_RF"].to_numpy() >= 0.5
    rows.append({"참가자": s, "FOG window": int(y.sum()),
                 "민감도(%)": 100 * pred[y].mean() if y.any() else np.nan,
                 "특이도(%)": 100 * (~pred[~y]).mean()})
tab = pd.DataFrame(rows)

def boot_ci(values, n_boot=5000):
    """참가자를 복원추출해 평균의 95% 구간을 구한다(window가 아니라 사람을 다시 뽑는다)."""
    v = np.asarray(values)
    means = [rng.choice(v, len(v), replace=True).mean() for _ in range(n_boot)]
    return np.percentile(means, [2.5, 97.5])

sens = tab["민감도(%)"].dropna(); spec = tab["특이도(%)"]
lo, hi = boot_ci(sens); lo2, hi2 = boot_ci(spec)
# 보고서에 붙일 Markdown 표(추가 패키지 없이 직접 만든다)
md = ["| " + " | ".join(tab.columns) + " |", "|" + "---|" * len(tab.columns)]
md += ["| " + " | ".join("-" if pd.isna(v) else (f"{v:.1f}" if isinstance(v, float) else str(v)) for v in r) + " |"
       for r in tab.itertuples(index=False)]
print("\n".join(md))
print(f"\n민감도: 참가자 평균 {sens.mean():.1f}% (참가자 부트스트랩 95% 구간 {lo:.1f}–{hi:.1f}, n = {len(sens)})")
print(f"특이도: 참가자 평균 {spec.mean():.1f}% (참가자 부트스트랩 95% 구간 {lo2:.1f}–{hi2:.1f}, n = {len(spec)})")
pooled = win["fog"].astype(bool); p = win["p_RF"] >= 0.5
print(f"참고 — window를 한데 모은 민감도 {100 * p[pooled].mean():.1f}%: 사람이 아니라 window를 세면 FOG가 많은 사람이 결과를 좌우한다")